# janus_01 – tanítás Kaggle-ön

Kaggle beállítás a jobb oldali panelen:
- **Accelerator**: GPU (T4 x2 vagy P100)
- **Internet**: On (a git clone miatt)
- **Input → Add data**: `asvpoof-2019-dataset` (awsaf49)

Utána *Run All*. A legjobb modell a `/kaggle/working/out/janus_best.pt` fájlba kerül, ami a futás végén letölthető az Output fülről.

In [ ]:
!nvidia-smi -L
!ls /kaggle/input

## 1. Kód behúzása GitHubról

In [ ]:
import os, sys
REPO = 'https://github.com/Csaszi05/janus_01.git'
if not os.path.exists('/kaggle/working/janus_01'):
    !git clone -q {REPO} /kaggle/working/janus_01
else:
    !cd /kaggle/working/janus_01 && git pull -q
%cd /kaggle/working/janus_01
!git log --oneline | head -3

In [ ]:
!pip install -q -r requirements.txt

## 2. Adat ellenőrzése

A Kaggle-adathalmaz gyökere alatt három dolognak kell lennie: `ASVspoof2019_LA_cm_protocols`, `ASVspoof2019_LA_train`, `ASVspoof2019_LA_dev`. Ha nem ez látszik, igazítsd a `DATA_ROOT`-ot.

In [ ]:
DATA_ROOT = '/kaggle/input/asvpoof-2019-dataset/LA/LA'
!ls {DATA_ROOT}

sys.path.insert(0, 'src')
from dataset import load_protocol
tr = load_protocol(DATA_ROOT, 'train'); dv = load_protocol(DATA_ROOT, 'dev')
print('train', len(tr), 'spoof arány', round(tr.y.mean(), 3))
print('dev  ', len(dv), 'spoof arány', round(dv.y.mean(), 3))
assert os.path.exists(tr.path[0]), tr.path[0]

## 3. Tanítás

- A mel-cache a `/tmp`-be megy, nem a `/kaggle/working`-be: kb. 4 GB, nem kell a kimenetbe menteni. Az első epoch ezért lassú (kiszámolja az összes spektrogramot), a továbbiak gyorsak.
- `--workers 2`: a Kaggle-kernel CPU-száma. T4 x2-nél lehet 4.
- A sor végén `*` jelzi, ha új legjobb dev-EER-t mentett.

In [ ]:
!python src/train.py \
    --data_root {DATA_ROOT} \
    --epochs 10 \
    --batch 64 \
    --lr 1e-3 \
    --workers 2 \
    --cache_dir /tmp/cache \
    --out /kaggle/working/out

## 4. Eredmény

In [ ]:
import torch
ck = torch.load('/kaggle/working/out/janus_best.pt', map_location='cpu', weights_only=False)
print('legjobb epoch:', ck['epoch'])
print('audio config:', ck['config']['audio'])
!ls -la /kaggle/working/out